# Set Library Path

In [1]:
.libPaths("/home/vhaghani/anaconda3/envs/jupyter_nb_R4.4.2/lib/R/library")

# Load Libraries

In [3]:
library(readxl)
library(ggplot2)
library(viridis)
library(dplyr)
library(glue)
library(tidyr)

# Load Data

In [8]:
# List all files
files <- list.files(path = "09_WGCNA_GO/MECP2", pattern = "_enrichr_results\\.xlsx$", full.names = TRUE)

# Extract module names from filenames and filter
module_names <- gsub("09_WGCNA_GO/MECP2/(.*)_enrichr_results\\.xlsx", "\\1", files)

# View the module names
print(module_names)

 [1] "antiquewhite4"   "bisque4"         "black"           "blue"           
 [5] "blue2"           "brown"           "brown2"          "brown4"         
 [9] "coral1"          "coral2"          "cyan"            "darkgreen"      
[13] "darkgrey"        "darkmagenta"     "darkolivegreen"  "darkolivegreen4"
[17] "darkorange"      "darkorange2"     "darkred"         "darkseagreen4"  
[21] "darkslateblue"   "darkturquoise"   "darkviolet"      "firebrick4"     
[25] "floralwhite"     "green"           "greenyellow"     "grey"           
[29] "grey60"          "honeydew1"       "indianred4"      "ivory"          
[33] "lavenderblush3"  "lightcoral"      "lightcyan"       "lightcyan1"     
[37] "lightgreen"      "lightpink3"      "lightpink4"      "lightsteelblue" 
[41] "lightsteelblue1" "lightyellow"     "magenta"         "magenta4"       
[45] "maroon"          "mediumorchid"    "mediumpurple2"   "mediumpurple3"  
[49] "midnightblue"    "navajowhite1"    "navajowhite2"    "orange"         

# Combined Dot Plot

In [9]:
# List of GO databases
databases <- c("GO_Biological_Process_2025", "GO_Cellular_Component_2025", "GO_Molecular_Function_2025",
               "KEGG_2021_Human", "Reactome_2022", "RNAseq_DiseaseGene_DrugSigs_GEO")

In [10]:
# Create abbreviation mapping
database_abbreviations <- c(
  "GO_Biological_Process_2025" = "GO BP",
  "GO_Cellular_Component_2025" = "GO CC", 
  "GO_Molecular_Function_2025" = "GO MF",
  "KEGG_2021_Human" = "KEGG",
  "Reactome_2022" = "REACTOME",
  "RNAseq_DiseaseGene_DrugSigs_GEO" = "GEO RNAseq"
)

In [11]:
# Initialize an empty dataframe for all GO data
all_data <- data.frame()

In [12]:
# Iterate over each module and read in the corresponding Excel file
for (file in files) {
  for (database in databases) {
    try({
      # Read the data from the Excel file
      df <- read_excel(file, sheet = database)
      
      if (nrow(df) == 0) {
        next  # Skip to the next database if no data
      }
      
      # Select the required columns and add module and database information
      df <- df %>%
        filter(Adjusted.P.value <= 0.05) %>%
        select(Term, Adjusted.P.value, Odds.Ratio) %>%
        mutate(Module = file, Database = database) %>%
        arrange(Adjusted.P.value)
      
      # Append to the dataframe
      all_data <- bind_rows(all_data, df)
      
    }, silent = TRUE)
  }
}

# Check if any data was read
if (nrow(all_data) == 0) {
  stop("No data read from any of the Excel files.")
}

# Display the combined dataframe
head(all_data)

,Term,Adjusted.P.value,Odds.Ratio,Module,Database
,<chr>,<dbl>,<dbl>,<chr>,<chr>
1,Cytoplasmic Translation (GO:0002181),6.065926e-10,5.796136,09_WGCNA_GO/MECP2/blue_enrichr_results.xlsx,GO_Biological_Process_2025
2,Macromolecule Biosynthetic Process (GO:0009059),1.111755e-05,3.122044,09_WGCNA_GO/MECP2/blue_enrichr_results.xlsx,GO_Biological_Process_2025
3,Translation (GO:0006412),1.979489e-05,2.778640,09_WGCNA_GO/MECP2/blue_enrichr_results.xlsx,GO_Biological_Process_2025
4,mRNA Processing (GO:0006397),1.979489e-05,2.794146,09_WGCNA_GO/MECP2/blue_enrichr_results.xlsx,GO_Biological_Process_2025
5,Regulation of Translation (GO:0006417),6.610848e-04,2.623698,09_WGCNA_GO/MECP2/blue_enrichr_results.xlsx,GO_Biological_Process_2025
6,Regulation of Transcription by RNA Polymerase II (GO:0006357),1.558810e-03,1.409382,09_WGCNA_GO/MECP2/blue_enrichr_results.xlsx,GO_Biological_Process_2025


In [13]:
# Format comparison name
all_data$Module <- gsub("09_WGCNA_GO/MECP2/", "", all_data$Module)
all_data$Module <- gsub("_enrichr_results.xlsx", "", all_data$Module)

# View
head(all_data)

,Term,Adjusted.P.value,Odds.Ratio,Module,Database
,<chr>,<dbl>,<dbl>,<chr>,<chr>
1,Cytoplasmic Translation (GO:0002181),6.065926e-10,5.796136,blue,GO_Biological_Process_2025
2,Macromolecule Biosynthetic Process (GO:0009059),1.111755e-05,3.122044,blue,GO_Biological_Process_2025
3,Translation (GO:0006412),1.979489e-05,2.778640,blue,GO_Biological_Process_2025
4,mRNA Processing (GO:0006397),1.979489e-05,2.794146,blue,GO_Biological_Process_2025
5,Regulation of Translation (GO:0006417),6.610848e-04,2.623698,blue,GO_Biological_Process_2025
6,Regulation of Transcription by RNA Polymerase II (GO:0006357),1.558810e-03,1.409382,blue,GO_Biological_Process_2025


In [14]:
# Define keywords
keywords <- c("neuro", "axon", "synapse", "receptor", "rho")

# Create a pattern for grepl
pattern <- paste(keywords, collapse = "|")

# Filter all_data
all_data <- all_data[grepl(pattern, all_data$Term, ignore.case = TRUE), ]

# View filtered data
head(all_data)

,Term,Adjusted.P.value,Odds.Ratio,Module,Database
,<chr>,<dbl>,<dbl>,<chr>,<chr>
50,Pathways of neurodegeneration,3.074743e-03,1.795632,blue,KEGG_2021_Human
56,Axon guidance,5.620342e-03,2.228073,blue,KEGG_2021_Human
66,Axon Guidance R-HSA-422475,3.173673e-09,2.356855,blue,Reactome_2022
69,Signaling By ROBO Receptors R-HSA-376176,5.014695e-09,3.358196,blue,Reactome_2022
81,"Signaling By Rho GTPases, Miro GTPases And RHOBTB3 R-HSA-9716542",5.268981e-07,1.975871,blue,Reactome_2022
82,Signaling By Rho GTPases R-HSA-194315,5.719758e-07,1.985095,blue,Reactome_2022


In [19]:
# Generate dot plot
if (nrow(all_data) > 0) {
  
  # Create database abbreviations and assign colors
  all_data <- all_data %>%
    mutate(
      DB_Abbrev = ifelse(Database %in% names(database_abbreviations), 
                         database_abbreviations[Database], Database),
      Module = factor(Module),
      # Create simple term label with database
      Simple_Term = paste(DB_Abbrev, Term, sep = ": ")
    )
  
  # Create a color mapping for databases
  unique_dbs <- unique(all_data$DB_Abbrev)
  db_colors <- setNames(RColorBrewer::brewer.pal(max(3, length(unique_dbs)), "Set3")[1:length(unique_dbs)], 
                       unique_dbs)
  
  # Sort terms by database first, then by term name
  all_data <- all_data %>%
    arrange(DB_Abbrev, Term)
  
  # Ensure terms are ordered properly
  term_order <- unique(all_data$Simple_Term)
  all_data$Simple_Term <- factor(all_data$Simple_Term, levels = term_order)
  
  dot_plot <- ggplot(all_data, aes(x = Module, y = Simple_Term, size = Odds.Ratio, fill = Adjusted.P.value)) +
    geom_point(shape = 21, alpha = 0.8) +
    scale_fill_viridis(name = 'Adjusted P-value', direction = -1) +
    scale_size_continuous(name = 'Odds Ratio', range = c(1, 8)) +
    xlab('') + ylab('') +
    labs(title = 'Significant Enrichment Terms for WGCNA Modules') +
    theme_minimal() +
    theme(
      panel.background = element_rect(fill = "white", color = NA),
      panel.grid.major = element_line(color = "grey90"),
      panel.grid.minor = element_blank(),
      axis.text.x = element_text(angle = 90, hjust = 1, vjust = 0.5),
      axis.text.y = element_text(size = 7),
      legend.position = "right"
    )
  
  # Save the plot
  ggsave(filename = "09_WGCNA_GO/combined_neurological_terms_dot_plot_adjpval.pdf", 
         plot = dot_plot, height = 15, width = 12)
}